# Experimental native paged attention: FlashInfer

Fresh Kaggle session: enable Internet and T4 GPU, then Run All. Source is embedded;
no upload, Git push or manual source edits are needed. Only GPU 0 is visible.
Download the final ZIP even on failure.

This changes only the explicitly selected decode attention kernel. PagedCore owns
the decoder, RoPE, FP16 K/V pool, block mapping and staged commits. The existing
backend remains the default while the candidate awaits T4 acceptance.

The cells install the frozen Torch 2.7.1+cu126 environment plus hash-verified
FlashInfer 0.2.5 JIT source and pinned Ninja. NVCC and a C++ compiler are required;
compilation support on Kaggle is unproven until this runs. Installation must not
change Torch. Missing dependencies or compilation failures export logs; there is
no automatic fallback or passing skip.

First check native attention with independent CPU FP64, poisoned fragmented pages,
lengths 1/15/16/17/49/2048, large scores, staged rollback and worker/stream ownership.
Then compare the current baseline and candidate at exactly 15 prefill + 1 decode.
Run the frozen single/mixed/fragmented/held-out paging suite with the candidate,
then a fixed 32-step cached trajectory (including an instrumentation control).
Logit/KV thresholds and exact greedy checks are unchanged. Retain old failures.

A passing notebook covers this experiment only. Whole-model independent quality,
service behavior, throughput and maximum-active workspace need separate evidence.


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import tempfile
import traceback
import zipfile
from pathlib import Path

from IPython.display import FileLink, display

FLASHINFER_URL = "https://files.pythonhosted.org/packages/b2/c4/9ec0f79e2480fc5c93307c4a1ac903e5cf33c551c0eaeb648196234b55af/flashinfer_python-0.2.5.tar.gz"
FLASHINFER_SHA256 = "990aa090ef781783e76b836696ece4efd23956f72b5696d622fc619a61162aef"
PAYLOAD = "..."
SNAPSHOT_SHA256 = "a0b306dcd4e462ee24e984164ec8033a9091bc07aee5a6d16d20d2f006318558"


output_override = os.environ.get("PAGEDCORE_EVIDENCE_WORKDIR")
working = (
    Path(output_override).expanduser().resolve()
    if output_override
    else Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd()
)
working.mkdir(parents=True, exist_ok=True)
run_root = Path(tempfile.mkdtemp(prefix="pagedcore-flashinfer-", dir=working))
source = run_root / "source"
evidence = run_root / "evidence"
evidence.mkdir()
payload = base64.b64decode(PAYLOAD)
assert hashlib.sha256(payload).hexdigest() == SNAPSHOT_SHA256
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(source)
identity = json.loads((source / "source-identity.json").read_text())
environment = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="0",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
    OMP_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    TOKENIZERS_PARALLELISM="false",
)
commands = []


def run(name, command, *, required=False):
    record = {"name": name, "argv": list(map(str, command)), "cwd": str(source)}
    commands.append(record)
    print(name, flush=True)
    with (evidence / (name + ".log")).open("w") as log:
        process = subprocess.Popen(
            command, cwd=source, env=environment, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    print(name + " is running; progress is saved in " + str(log.name), flush=True)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            record["interrupted"] = True
            raise
    record["returncode"] = process.returncode
    (evidence / "commands.json").write_text(json.dumps(commands, indent=2))
    print(name, "exit", process.returncode, flush=True)
    if required and process.returncode:
        raise RuntimeError(name + " failed; inspect its retained log")


def verify_source():
    checks = {
        name: hashlib.sha256((source / name).read_bytes()).hexdigest()
        for name in identity["sha256"]
    }
    assert checks == identity["sha256"], "source changed after notebook generation"
    return checks


setup_ok = False
try:
    verify_source()
    run("setup-uv", [sys.executable, "-m", "pip", "install", "uv==0.11.27"], required=True)
    run(
        "setup-frozen",
        [
            sys.executable,
            "-m",
            "uv",
            "sync",
            "--frozen",
            "--extra",
            "cuda",
            "--extra",
            "dev",
            "--python",
            (source / ".python-version").read_text().strip(),
        ],
        required=True,
    )
    python = source / ".venv/bin/python"
    run(
        "resolved-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
    )
    run("gpu-inventory", ["nvidia-smi", "-q"])

    target_code = "import json,sys,torch; from pathlib import Path; from pagedcore.cli import collect_environment_diagnostics; assert torch.cuda.is_available() and torch.cuda.device_count()==1; assert 'T4' in torch.cuda.get_device_name(0); assert torch.__version__=='2.7.1+cu126'; p=torch.cuda.get_device_properties(0); Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,name=p.name,uuid=str(p.uuid),capability=[p.major,p.minor],visible_devices=torch.cuda.device_count(),total_memory=p.total_memory,environment=collect_environment_diagnostics()),indent=2))"
    run("target", [str(python), "-c", target_code, str(evidence / "target.json")], required=True)
    run(
        "checkpoint-verify",
        [
            str(python),
            "-m",
            "scripts.verify_owned_checkpoint",
            "--output",
            str(evidence / "environment-manifest.json"),
        ],
        required=True,
    )

    environment["TORCH_CUDA_ARCH_LIST"] = "7.5"
    environment["MAX_JOBS"] = "2"
    # Record the actual SDK/compiler; JIT compatibility must be proved by execution.
    run("nvcc", ["nvcc", "--version"], required=True)
    run("compiler", ["c++", "--version"], required=True)
    run(
        "setup-ninja",
        [
            sys.executable,
            "-m",
            "uv",
            "pip",
            "install",
            "--python",
            str(python),
            "--no-deps",
            "ninja==1.11.1.4",
        ],
        required=True,
    )
    download = """
import hashlib, sys, urllib.request
from pathlib import Path
data = urllib.request.urlopen(sys.argv[1], timeout=120).read()
assert hashlib.sha256(data).hexdigest() == sys.argv[2], 'FlashInfer source checksum mismatch'
Path(sys.argv[3]).write_bytes(data)
"""
    artifact = evidence / "flashinfer_python-0.2.5.tar.gz"
    run(
        "download-flashinfer",
        [str(python), "-c", download, FLASHINFER_URL, FLASHINFER_SHA256, str(artifact)],
        required=True,
    )
    run(
        "setup-flashinfer",
        [
            sys.executable,
            "-m",
            "uv",
            "pip",
            "install",
            "--python",
            str(python),
            "--no-deps",
            "--no-build-isolation",
            str(artifact),
        ],
        required=True,
    )
    run(
        "candidate-dependencies",
        [sys.executable, "-m", "uv", "pip", "freeze", "--python", str(python)],
        required=True,
    )
    run(
        "candidate-dependency-check",
        [sys.executable, "-m", "uv", "pip", "check", "--python", str(python)],
        required=True,
    )
    installed = """
import json, sys, torch
from importlib.metadata import version
from pathlib import Path
assert torch.__version__ == '2.7.1+cu126', 'candidate install changed Torch'
assert version('flashinfer-python') == '0.2.5'
Path(sys.argv[1]).write_text(json.dumps(dict(torch=torch.__version__,
    flashinfer=version('flashinfer-python'), ninja=version('ninja'),
    cuda=torch.version.cuda, source_sha256=sys.argv[2]), indent=2))
"""
    run(
        "candidate-identity",
        [str(python), "-c", installed, str(evidence / "candidate.json"), FLASHINFER_SHA256],
        required=True,
    )

    setup_ok = True
except BaseException:
    (evidence / "setup-failure.txt").write_text(traceback.format_exc())
    print("Setup incomplete; run the final cell to download its logs.")

In [ ]:
checks_ok = False
try:
    assert setup_ok, "Setup did not finish"
    run("lint", [str(python), "-m", "ruff", "check", "."], required=True)
    run("format", [str(python), "-m", "ruff", "format", "--check", "."], required=True)
    run("types", [str(python), "-m", "mypy", "--strict", "src/pagedcore"], required=True)
    run(
        "cpu-tests",
        [
            str(python),
            "-m",
            "pytest",
            "-m",
            "not gpu and not benchmark",
            "-ra",
            "--junitxml=" + str(evidence / "cpu-tests.xml"),
        ],
        required=True,
    )
    checks_ok = True
except BaseException:
    (evidence / "checks-failure.txt").write_text(traceback.format_exc())
    print("Checks incomplete; keep running to the export cell.")

In [ ]:
kernel_ok = False
try:
    assert checks_ok, "CPU checks did not finish"
    run(
        "native-kernel-tests",
        [
            str(python),
            "-m",
            "pytest",
            "tests/gpu/test_flashinfer_cuda.py",
            "--decode-backend=flashinfer",
            "-v",
            "-ra",
            "--junitxml=" + str(evidence / "native-kernel-tests.xml"),
        ],
        required=True,
    )
    kernel_ok = True
except BaseException:
    (evidence / "kernel-failure.txt").write_text(traceback.format_exc())
    print("Native kernel checks incomplete; export the failure logs.")

In [ ]:
try:
    assert kernel_ok, "Native kernel checks did not pass"
    for backend in ("pagedcore", "flashinfer"):
        run(
            "focused-15-plus-1-" + backend,
            [
                str(python),
                "-m",
                "scripts.probe_two_token_decode",
                "--prompt-tokens",
                "15",
                "--single-only",
                "--attention-backend",
                backend,
                "--output-dir",
                str(evidence / ("focused-" + backend)),
            ],
            required=True,
        )
    # A focused numerical failure is retained by the diagnostic; it must not hide the suite.
    environment["PAGEDCORE_ACCURACY_DIR"] = str(evidence / "raw-logits")
    run(
        "gpu-regressions",
        [
            str(python),
            "-m",
            "pytest",
            "-m",
            "gpu",
            "--decode-backend=flashinfer",
            "-v",
            "-ra",
            "--junitxml=" + str(evidence / "gpu-tests.xml"),
        ],
    )
    run("accuracy-export", [str(python), "-m", "scripts.export_paged_accuracy", str(evidence)])
    run(
        "candidate-cached-trajectory",
        [
            str(python),
            "-m",
            "scripts.check_checkpoint_trajectory",
            "--attention-backend",
            "flashinfer",
            "--output-dir",
            str(evidence / "trajectory"),
        ],
        required=True,
    )
    summary = json.loads((evidence / "accuracy-summary.json").read_text())
    print("Frozen paging accuracy:", summary["status"])
    print("This is an experimental backend result; P5.3 needs review of the exported evidence.")
except BaseException:
    (evidence / "integration-failure.txt").write_text(traceback.format_exc())
    print("Integration incomplete; export the failure and partial evidence.")

In [ ]:
try:
    (evidence / "source-after.json").write_text(json.dumps(verify_source(), indent=2))
except BaseException:
    (evidence / "source-verification-failure.txt").write_text(traceback.format_exc())
(evidence / "commands.json").write_text(json.dumps(commands, indent=2))
(evidence / "notebook-run.json").write_text(
    json.dumps(
        {
            "snapshot_sha256": SNAPSHOT_SHA256,
            "base_revision": identity["base_revision"],
            "scope": "Paged-decoder accuracy validation, not full release acceptance",
            "environment_overrides": {
                key: environment[key]
                for key in (
                    "CUDA_VISIBLE_DEVICES",
                    "CUBLAS_WORKSPACE_CONFIG",
                    "OMP_NUM_THREADS",
                    "MKL_NUM_THREADS",
                )
            },
        },
        indent=2,
    )
)
files = {
    path.relative_to(evidence).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in evidence.rglob("*")
    if path.is_file()
}
(evidence / "checksums.json").write_text(json.dumps(files, indent=2))
archive_path = working / (run_root.name + ".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("source-snapshot.zip", payload)
    for path in evidence.rglob("*"):
        if path.is_file():
            archive.write(path, "evidence/" + path.relative_to(evidence).as_posix())
print("DOWNLOAD", archive_path, "bytes", archive_path.stat().st_size, flush=True)
display(FileLink(str(archive_path)))